# Notebook: Qualitative Evaluation (Expert Analysis) -- Experiment 1

In this notebook, the qualitative evaluation of Experiment 1 is conducted for the question-generation behind RQ1 and RQ3.

The notebook uses one evaluation setup with exactly 3 experts.

All rating categories are interpreted on a 1-5 Likert scale. Question-related and answer-related ratings are analyzed separately and as combined total scores.

## Initial Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
from statsmodels.stats.inter_rater import fleiss_kappa
import pingouin as pg

warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.notebook_repr_html', True)

LIKERT_MIN = 1
LIKERT_MAX = 5
LIKERT_RANGE = (LIKERT_MIN, LIKERT_MAX)
EXPERT_IDS = [1, 2, 3]
ANALYSIS_SUFFIX = 'experts'

EXP1_QUESTION_COLS = [
    'q_relevance',
    'q_clarity',
    'q_answerability',
    'q_challenging',
    'q_value',
    'q_language',
]
EXP1_ANSWER_COLS = [
    'a_clarity',
    'a_language',
    'a_correctness',
]
EXP1_NUMERIC_COLS = EXP1_QUESTION_COLS + EXP1_ANSWER_COLS
EXP1_SCORE_COLS = ['question_total_score', 'answer_total_score', 'total_score']
SCORE_RANGES = {
    'question_total_score': (len(EXP1_QUESTION_COLS) * LIKERT_MIN, len(EXP1_QUESTION_COLS) * LIKERT_MAX),
    'answer_total_score': (len(EXP1_ANSWER_COLS) * LIKERT_MIN, len(EXP1_ANSWER_COLS) * LIKERT_MAX),
    'total_score': (len(EXP1_NUMERIC_COLS) * LIKERT_MIN, len(EXP1_NUMERIC_COLS) * LIKERT_MAX),
}

DISPLAY_LABELS = {
    'q_relevance': 'qRelevance',
    'q_clarity': 'qClarity',
    'q_answerability': 'qAnswerability',
    'q_challenging': 'qChallenging',
    'q_value': 'qValue',
    'q_language': 'qLanguage',
    'a_clarity': 'aClarity',
    'a_language': 'aLanguage',
    'a_correctness': 'aCorrectness',
    'question_total_score': 'qTotal',
    'answer_total_score': 'aTotal',
    'total_score': 'total',
}

BASE_PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
qualitative_base_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/qualitative/exp1")
sampled_hints_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/sampled/exp1_sampled.csv")
output_base_path = os.path.join(BASE_PROJECT_PATH, "40_evaluation/exp1/qualitative")
output_tables_path = os.path.join(output_base_path, "tables")
output_plots_path = os.path.join(output_base_path, "plots")

for path in [output_tables_path, output_plots_path]:
    os.makedirs(path, exist_ok=True)

LABEL_MAPPING = {
    'anthropic': 'Anthropic',
    'openai': 'OpenAI',
    'deepseek': 'DeepSeek',
    'xai': 'xAI',
    'google': 'Google',
    'mcq': 'MCQ',
    'open_ended': 'Open-Ended',
    'layer1': 'Layer 1',
    'layer2': 'Layer 2',
    'layer3': 'Layer 3',
    'layer4': 'Layer 4',
    'layer5': 'Layer 5',
    'layer6': 'Layer 6',
    'layer7': 'Layer 7',
}

tables = {}
plots = {}

print("Setup completed successfully")
print(f"Output tables: {output_tables_path}")
print(f"Output plots: {output_plots_path}")
print(f"Experts configured: {EXPERT_IDS}")
print("Scale: 1-5 Likert per rating category")

In [ ]:
# Utility Functions

def normalize_sample_id(series):
    return series.astype(str).str.extract(r'(\d+)')[0].str.zfill(3)


def create_seaborn_boxplot(data, x, y, ax, title, ylabel, xlabel, scale_range=None):
    plot_df = data[[x, y]].dropna().copy()
    if plot_df.empty:
        ax.set_title(f'{title} - No data', fontsize=12, fontweight='bold')
        ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        return

    colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']
    order = sorted(plot_df[x].dropna().unique(), key=lambda value: str(value))
    palette = colors[:len(order)]

    sns.boxplot(
        data=plot_df,
        x=x,
        y=y,
        order=order,
        ax=ax,
        palette=palette,
        medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'},
    )

    for i, value in enumerate(order):
        mean_val = plot_df.loc[plot_df[x] == value, y].mean()
        ax.scatter(i, mean_val, color='red', marker='D', s=50, zorder=3, edgecolor='darkred', linewidth=1)

    ax.set_xticklabels([LABEL_MAPPING.get(str(label), str(label)) for label in order], rotation=0)

    if scale_range is not None:
        padding = 0.1 if scale_range[1] <= 5 else 0.5
        ax.set_ylim(scale_range[0] - padding, scale_range[1] + padding)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11, labelpad=15)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=10)
    ax.grid(True, alpha=0.3)


def create_heatmap(data, index_col, column_col, value_col, ax, title, cbar_label, scale_range=None):
    heatmap_mean = data.groupby([index_col, column_col])[value_col].mean().unstack()
    heatmap_std = data.groupby([index_col, column_col])[value_col].std().unstack()

    annot_matrix = heatmap_mean.copy().astype(object)
    for i in range(len(heatmap_mean.index)):
        for j in range(len(heatmap_mean.columns)):
            mean_val = heatmap_mean.iloc[i, j]
            std_val = heatmap_std.iloc[i, j]
            if pd.notna(mean_val) and pd.notna(std_val):
                annot_matrix.iloc[i, j] = f'{mean_val:.2f}\n({std_val:.2f})'
            elif pd.notna(mean_val):
                annot_matrix.iloc[i, j] = f'{mean_val:.2f}'
            else:
                annot_matrix.iloc[i, j] = ''

    yticklabels = [LABEL_MAPPING.get(str(label), str(label).title()) for label in heatmap_mean.index]
    xticklabels = [LABEL_MAPPING.get(str(label), str(label).title()) for label in heatmap_mean.columns]

    heatmap_kwargs = {
        'annot': annot_matrix,
        'fmt': '',
        'cmap': 'RdYlBu_r',
        'square': True,
        'linewidths': 0.5,
        'cbar_kws': {'shrink': 0.8, 'label': cbar_label},
        'annot_kws': {'size': 10, 'weight': 'bold'},
        'xticklabels': xticklabels,
        'yticklabels': yticklabels,
        'ax': ax,
    }

    if scale_range is not None:
        heatmap_kwargs['vmin'] = scale_range[0]
        heatmap_kwargs['vmax'] = scale_range[1]
        heatmap_kwargs['center'] = (scale_range[0] + scale_range[1]) / 2
    elif not heatmap_mean.empty:
        heatmap_kwargs['center'] = heatmap_mean.mean().mean()

    sns.heatmap(heatmap_mean, **heatmap_kwargs)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel(LABEL_MAPPING.get(column_col, column_col.replace('_', ' ').title()), fontsize=12, fontweight='bold')
    ax.set_ylabel(LABEL_MAPPING.get(index_col, index_col.replace('_', ' ').title()), fontsize=12, fontweight='bold')


print("Utility functions loaded successfully")

In [ ]:
def enrich_exp1_metadata(df):
    if not os.path.exists(sampled_hints_path):
        raise FileNotFoundError(f"Sampled file not found at {sampled_hints_path}")

    sampled_df = pd.read_csv(sampled_hints_path).copy()
    sampled_df['sample_id'] = [f'{i + 1:03d}' for i in range(len(sampled_df))]

    df = df.copy()
    df['sample_id'] = normalize_sample_id(df['sample_id'])

    exp1_df = pd.merge(
        df,
        sampled_df[['sample_id', 'llm', 'layer', 'question_type', 'bloom_idx']],
        on='sample_id',
        how='left',
        suffixes=('', '_meta'),
    )

    for col in ['layer', 'question_type']:
        meta_col = f'{col}_meta'
        if meta_col in exp1_df.columns:
            if col in exp1_df.columns:
                exp1_df[col] = exp1_df[col].combine_first(exp1_df[meta_col])
            else:
                exp1_df[col] = exp1_df[meta_col]
            exp1_df = exp1_df.drop(columns=[meta_col])

    if 'layer' in exp1_df.columns:
        exp1_df['layer'] = pd.to_numeric(exp1_df['layer'], errors='coerce').astype('Int64')
        exp1_df['input_source'] = np.where(exp1_df['layer'].notna(), 'layer' + exp1_df['layer'].astype(str), pd.NA)

    if 'question_type' in exp1_df.columns:
        valid_types = ['mcq', 'open_ended']
        exp1_df['question_type'] = exp1_df['question_type'].where(exp1_df['question_type'].isin(valid_types), pd.NA)

    return exp1_df

In [ ]:
def load_all_experts_data():
    experts_data = {}
    all_expert_dfs = []

    for expert_num in EXPERT_IDS:
        expert_key = f'expert_{expert_num}'
        expert_file = os.path.join(qualitative_base_path, f'exp1_eval_e{expert_num}.csv')

        if not os.path.exists(expert_file):
            print(f"Skip {expert_key}: file missing ({expert_file})")
            continue

        expert_df = pd.read_csv(expert_file)

        if expert_df.empty:
            print(f"Skip {expert_key}: empty CSV")
            continue

        first_row = expert_df.iloc[0]
        empty_count = first_row.apply(
            lambda v: pd.isna(v) or (isinstance(v, str) and v.strip() == '')
        ).sum()

        if empty_count >= 4:
            print(f"Skip {expert_key}: first data row has {empty_count} empty fields (>=4)")
            continue

        expert_df = enrich_exp1_metadata(expert_df)
        expert_df['expert'] = expert_key
        experts_data[expert_key] = expert_df.copy()
        all_expert_dfs.append(expert_df)

    if not all_expert_dfs:
        raise ValueError("No usable expert CSV found.")

    exp1_df = pd.concat(all_expert_dfs, ignore_index=True)
    return exp1_df, experts_data

In [ ]:
# def load_all_experts_data():
#     experts_data = {}
#     all_expert_dfs = []

#     for expert_num in EXPERT_IDS:
#         expert_key = f'expert_{expert_num}'
#         expert_file = os.path.join(qualitative_base_path, f'exp1_eval_e{expert_num}.csv')

#         if not os.path.exists(expert_file):
#             raise FileNotFoundError(f'Missing required file: {expert_file}')

#         expert_df = pd.read_csv(expert_file)
#         expert_df = enrich_exp1_metadata(expert_df)
#         expert_df['expert'] = expert_key
#         experts_data[expert_key] = expert_df.copy()
#         all_expert_dfs.append(expert_df)

#     exp1_df = pd.concat(all_expert_dfs, ignore_index=True)
#     return exp1_df, experts_data

In [ ]:
def clean_numeric_data(df, numeric_cols):
    invalid_values = [
        '??', '???', '?', '????', '', ' ', 'nan', 'NaN', 'NULL', 'null',
        'None', 'NONE', 'n/a', 'N/A', '#N/A', '#NULL!',
        'undefined', 'UNDEFINED', '-', '--', '---', '-99',
    ]

    df = df.copy()
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace(invalid_values, np.nan)
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.loc[(df[col] < LIKERT_MIN) | (df[col] > LIKERT_MAX), col] = np.nan
    return df


def calculate_exp1_scores(df):
    df = df.copy()
    df['question_total_score'] = df[EXP1_QUESTION_COLS].sum(axis=1, min_count=len(EXP1_QUESTION_COLS))
    df['answer_total_score'] = df[EXP1_ANSWER_COLS].sum(axis=1, min_count=len(EXP1_ANSWER_COLS))
    df['total_score'] = df[EXP1_NUMERIC_COLS].sum(axis=1, min_count=len(EXP1_NUMERIC_COLS))
    return df

In [ ]:
def calculate_agreement_reliability(expert_dfs_dict, criteria_cols, min_value, max_value):
    import scipy.stats as ss
    results = []

    expert_keys = sorted(expert_dfs_dict.keys())
    if len(expert_keys) < 2:
        print("Need at least 2 experts for agreement analysis")
        return pd.DataFrame()

    allowed = set(range(min_value, max_value + 1))

    def _fleiss_kappa_from_ratings_array(ratings_arr: np.ndarray):
        fleiss_table_local = np.zeros((len(ratings_arr), max_value - min_value + 1), dtype=int)
        for i, item_ratings in enumerate(ratings_arr):
            for r in item_ratings:
                fleiss_table_local[i, r - min_value] += 1
        return float(fleiss_kappa(fleiss_table_local))

    def _fleiss_kappa_permutation_pvalue(ratings_arr: np.ndarray, n_permutations: int = 2000, random_state: int = 42):
        observed = _fleiss_kappa_from_ratings_array(ratings_arr)
        rng = np.random.default_rng(random_state)
        permuted = np.empty_like(ratings_arr)
        extreme = 0
        for _ in range(n_permutations):
            for col in range(ratings_arr.shape[1]):
                permuted[:, col] = rng.permutation(ratings_arr[:, col])
            stat = _fleiss_kappa_from_ratings_array(permuted)
            if abs(stat) >= abs(observed):
                extreme += 1
        return (extreme + 1) / (n_permutations + 1)

    def _unique_valid_rating(series: pd.Series):
        s = pd.to_numeric(series, errors="coerce").dropna()
        s = s[s.isin(allowed)]
        uniques = pd.unique(s)
        if len(uniques) == 1:
            return float(uniques[0])
        return np.nan

    collapsed = {}
    base_common_ids = None

    for expert_key in expert_keys:
        df = expert_dfs_dict[expert_key].copy()

        if "sample_id" not in df.columns:
            print(f"Skip {expert_key}: missing sample_id column")
            continue

        df["sample_id"] = normalize_sample_id(df["sample_id"])
        df = clean_numeric_data(df, criteria_cols)

        dup_counts = df.groupby("sample_id").size()
        dup_ids = dup_counts[dup_counts > 1]
        if len(dup_ids) > 0:
            print(f"Note: {expert_key} has {len(dup_ids)} duplicate sample_id(s); collapsing to unique ratings per criterion.")

        keep_cols = ["sample_id"] + [c for c in criteria_cols if c in df.columns]
        df = df[keep_cols].copy()

        collapsed_df = df.groupby("sample_id", dropna=True).agg({c: _unique_valid_rating for c in keep_cols if c != "sample_id"})
        collapsed[expert_key] = collapsed_df

        ids = set(collapsed_df.index.dropna().astype(str))
        base_common_ids = ids if base_common_ids is None else (base_common_ids & ids)

    if not collapsed or not base_common_ids:
        print("No common sample_ids found across experts")
        return pd.DataFrame()

    base_common_ids = sorted(base_common_ids)
    print(f"Analyzing {len(collapsed)} experts with {len(base_common_ids)} common samples")

    for criterion in criteria_cols:
        if any(criterion not in collapsed[k].columns for k in collapsed.keys()):
            continue

        valid_ids = set(base_common_ids)
        for expert_key in collapsed.keys():
            s = collapsed[expert_key][criterion]
            valid_ids &= set(s[s.notna()].index.astype(str))

        valid_ids = sorted(valid_ids)

        if len(valid_ids) < 2:
            continue

        ratings = []
        for sample_id in valid_ids:
            item = []
            ok = True
            for expert_key in collapsed.keys():
                val = collapsed[expert_key].loc[sample_id, criterion]
                if pd.isna(val) or (val not in allowed):
                    ok = False
                    break
                item.append(int(val))
            if ok:
                ratings.append(item)

        if len(ratings) < 2:
            continue

        ratings_array = np.asarray(ratings, dtype=int)

        # 1. Fleiss' Kappa calculation (+ permutation p-value)
        kappa = _fleiss_kappa_from_ratings_array(ratings_array)
        fleiss_k_pval = _fleiss_kappa_permutation_pvalue(ratings_array)
        level = (
            "Poor" if kappa < 0 else
            "Slight" if kappa < 0.2 else
            "Fair" if kappa < 0.4 else
            "Moderate" if kappa < 0.6 else
            "Substantial" if kappa < 0.8 else
            "Almost Perfect"
        )

        # 2. Kendall's W calculation
        n_items, m_raters = ratings_array.shape
        ranks = np.zeros_like(ratings_array, dtype=float)
        tie_corr = 0
        for j in range(m_raters):
            ranks[:, j] = ss.rankdata(ratings_array[:, j])
            _, counts = np.unique(ratings_array[:, j], return_counts=True)
            t = counts[counts > 1]
            tie_corr += np.sum(t**3 - t)

        R_i = np.sum(ranks, axis=1)
        S = np.sum((R_i - np.mean(R_i))**2)
        denom = (m_raters**2 * (n_items**3 - n_items)) - (m_raters * tie_corr)
        kendalls_w = (12 * S) / denom if denom != 0 else np.nan
        kendalls_chi2 = (m_raters * (n_items - 1) * kendalls_w) if pd.notna(kendalls_w) else np.nan
        kendalls_pval = ss.chi2.sf(kendalls_chi2, df=n_items - 1) if pd.notna(kendalls_chi2) else np.nan

        if criterion in EXP1_QUESTION_COLS:
            rating_domain = "Question"
        elif criterion in EXP1_ANSWER_COLS:
            rating_domain = "Answer"
        else:
            rating_domain = "Other"

        results.append({
            "Criterion": criterion,
            "Fleiss_K": round(float(kappa), 3),
            "Fleiss_K_pval": round(float(fleiss_k_pval), 4),
            "Agreement": level,
            "Kendalls_W": round(float(kendalls_w), 3),
            "Kendalls_W_pval": round(float(kendalls_pval), 4) if pd.notna(kendalls_pval) else np.nan,
            "N_Items": int(len(ratings)),
            "N_Raters": int(len(collapsed)),
            "Mean": round(float(ratings_array.mean()), 2),
            "Std": round(float(ratings_array.std()), 2),
            "Scale": f"{min_value}-{max_value}",
            "Domain": rating_domain,
        })

    return pd.DataFrame(results)


def calculate_icc_3_1(expert_dfs_dict, criteria_cols, min_value, max_value):
    results = []

    expert_keys = sorted(expert_dfs_dict.keys())
    if len(expert_keys) < 2:
        print('Need at least 2 experts for ICC analysis')
        return pd.DataFrame()

    selected_expert_keys = expert_keys[:3]
    if len(selected_expert_keys) < 3:
        print('ICC(3,1) requires exactly 3 selected raters in this notebook')
        return pd.DataFrame()

    allowed = set(range(min_value, max_value + 1))

    def _unique_valid_rating(series: pd.Series):
        s = pd.to_numeric(series, errors='coerce').dropna()
        s = s[s.isin(allowed)]
        uniques = pd.unique(s)
        if len(uniques) == 1:
            return float(uniques[0])
        return np.nan

    collapsed = {}
    base_common_ids = None

    for expert_key in selected_expert_keys:
        df = expert_dfs_dict[expert_key].copy()
        if 'sample_id' not in df.columns:
            continue

        df['sample_id'] = normalize_sample_id(df['sample_id'])
        df = clean_numeric_data(df, criteria_cols)

        keep_cols = ['sample_id'] + [c for c in criteria_cols if c in df.columns]
        df = df[keep_cols].copy()
        collapsed_df = df.groupby('sample_id', dropna=True).agg({
            c: _unique_valid_rating for c in keep_cols if c != 'sample_id'
        })
        collapsed[expert_key] = collapsed_df

        ids = set(collapsed_df.index.dropna().astype(str))
        base_common_ids = ids if base_common_ids is None else (base_common_ids & ids)

    if len(collapsed) < 3 or not base_common_ids:
        print('No sufficient common sample_ids found for ICC across 3 selected experts')
        return pd.DataFrame()

    for criterion in criteria_cols:
        if any(criterion not in collapsed[k].columns for k in selected_expert_keys):
            continue

        valid_ids = set(base_common_ids)
        for expert_key in selected_expert_keys:
            s = collapsed[expert_key][criterion]
            valid_ids &= set(s[s.notna()].index.astype(str))

        valid_ids = sorted(valid_ids)
        if len(valid_ids) < 2:
            continue

        long_rows = []
        for sample_id in valid_ids:
            for expert_key in selected_expert_keys:
                val = collapsed[expert_key].loc[sample_id, criterion]
                if pd.isna(val) or (val not in allowed):
                    continue
                long_rows.append({
                    'targets': sample_id,
                    'raters': expert_key,
                    'ratings': float(val),
                })

        icc_input = pd.DataFrame(long_rows)
        if icc_input.empty:
            continue

        per_target_counts = icc_input.groupby('targets')['raters'].nunique()
        complete_targets = per_target_counts[per_target_counts == len(selected_expert_keys)].index
        icc_input = icc_input[icc_input['targets'].isin(complete_targets)].copy()

        if icc_input['targets'].nunique() < 2:
            continue

        icc_table = pg.intraclass_corr(
            data=icc_input,
            targets='targets',
            raters='raters',
            ratings='ratings',
            nan_policy='omit',
        )
        icc_row = icc_table.loc[icc_table['Type'] == 'ICC3']
        if icc_row.empty:
            continue

        icc_row = icc_row.iloc[0]
        ci95 = icc_row.get('CI95', np.nan)
        if isinstance(ci95, (list, tuple, np.ndarray, pd.Series)):
            ci95_vals = pd.Series(ci95).dropna().tolist()
            ci95_str = str(ci95_vals) if len(ci95_vals) > 0 else np.nan
        elif pd.isna(ci95):
            ci95_str = np.nan
        else:
            ci95_str = str(ci95)

        if criterion in EXP1_QUESTION_COLS:
            rating_domain = 'Question'
        elif criterion in EXP1_ANSWER_COLS:
            rating_domain = 'Answer'
        else:
            rating_domain = 'Other'

        results.append({
            'Criterion': criterion,
            'ICC3_1': round(float(icc_row['ICC']), 3),
            'ICC3_1_F': round(float(icc_row['F']), 3),
            'ICC3_1_df1': int(icc_row['df1']),
            'ICC3_1_df2': int(icc_row['df2']),
            'ICC3_1_pval': round(float(icc_row['pval']), 4),
            'ICC3_1_CI95': ci95_str,
            'N_Items': int(icc_input['targets'].nunique()),
            'N_Raters': int(len(selected_expert_keys)),
            'Scale': f'{min_value}-{max_value}',
            'Rating_Domain': rating_domain,
        })

    return pd.DataFrame(results)


print("Agreement helper loaded successfully")

## Data Loading and Configuration

In [ ]:
print(f"Loading unified expert data (Experts {EXPERT_IDS[0]}-{EXPERT_IDS[-1]})")
exp1_df, experts_data = load_all_experts_data()
agreement_available = len(experts_data) >= 2
analysis_suffix = ANALYSIS_SUFFIX

exp1_df = clean_numeric_data(exp1_df, EXP1_NUMERIC_COLS)
exp1_df = calculate_exp1_scores(exp1_df)
criteria = EXP1_NUMERIC_COLS + EXP1_SCORE_COLS

exp1_filled = exp1_df[EXP1_NUMERIC_COLS].notna().sum().sum()
exp1_total = len(exp1_df) * len(EXP1_NUMERIC_COLS)
completion_pct = (100 * exp1_filled / exp1_total) if exp1_total else 0

print(f"\nData loaded successfully!")
print(f"Experiment 1: {len(exp1_df)} rows, {completion_pct:.1f}% rating completion")
print(f"Experts loaded: {list(experts_data.keys())}")

if 'llm' in exp1_df.columns:
    print(f"\nDistribution:")
    print(f"  LLMs: {list(exp1_df['llm'].dropna().unique())}")
if 'question_type' in exp1_df.columns:
    print(f"  Question Types: {list(exp1_df['question_type'].dropna().unique())}")
if 'layer' in exp1_df.columns:
    print(f"  Layers (OSI): {sorted(exp1_df['layer'].dropna().unique())}")

if 'layer' in exp1_df.columns and 'input_source' not in exp1_df.columns:
    exp1_df['input_source'] = 'layer' + exp1_df['layer'].astype(str)

# Experiment 1: OSI Layer-Based Analysis

Analysis of LLM question generation quality using OSI layer source materials.

In [ ]:
print("EXPERIMENT 1 - DESCRIPTIVE STATISTICS")
print("="*60)

criteria = EXP1_NUMERIC_COLS + EXP1_SCORE_COLS

exp1_stats = exp1_df[criteria].describe().round(2)
tables[f'exp1_overall_stats_{analysis_suffix}'] = exp1_stats
print("\nOverall Statistics:")
display(exp1_stats)

In [ ]:
# Statistics by LLM
exp1_llm_stats = exp1_df.groupby('llm')[criteria].agg(['mean', 'std', 'median', 'count']).round(2)
tables[f'exp1_llm_stats_{analysis_suffix}'] = exp1_llm_stats
print("\nStatistics by LLM:")
display(exp1_llm_stats)

In [ ]:
# LLM ranking with combined mean of the 3 total score columns
ranking_cols = ['question_total_score', 'answer_total_score', 'total_score']
exp1_llm_means = exp1_df.groupby('llm')[ranking_cols].mean().round(2)
exp1_llm_means['combined_mean'] = exp1_llm_means.mean(axis=1)
exp1_llm_overall = exp1_llm_means.sort_values('combined_mean', ascending=False)
tables[f'exp1_llm_ranking_{analysis_suffix}'] = exp1_llm_overall

print("\nOverall LLM Ranking:")
display(exp1_llm_overall)

In [ ]:
# Statistics by Question Type
exp1_question_type_stats = exp1_df.groupby('question_type')[criteria].agg(['mean', 'std', 'median', 'count']).round(2)
tables[f'exp1_question_type_stats_{analysis_suffix}'] = exp1_question_type_stats
print("\nStatistics by Question Type:")
display(exp1_question_type_stats)

In [ ]:
# Statistics by Bloom Level
exp1_bloom_stats = exp1_df.groupby('bloom_idx')[criteria].agg(['mean', 'std', 'median', 'count']).round(2)
tables[f'exp1_bloom_stats_{analysis_suffix}'] = exp1_bloom_stats
print("\nStatistics by Bloom Level:")
display(exp1_bloom_stats)

In [ ]:
question_plot_metrics = EXP1_QUESTION_COLS + ['question_total_score']

fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(2, 8)

axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[0, 4:6]),
    fig.add_subplot(gs[0, 6:8]),
    fig.add_subplot(gs[1, 1:3]),
    fig.add_subplot(gs[1, 3:5]),
    fig.add_subplot(gs[1, 5:7]),
]

plots[f'exp1_question_bundle_question_type_{analysis_suffix}'] = fig

for i, criterion in enumerate(question_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp1_df,
        'question_type',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'Question Type',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 1: Question-based Criteria', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
answer_plot_metrics = EXP1_ANSWER_COLS + ['answer_total_score']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
plots[f'exp1_answer_bundle_question_type_{analysis_suffix}'] = fig

for i, criterion in enumerate(answer_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp1_df,
        'question_type',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'Question Type',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 1: Answer-based Criteria', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()


In [ ]:
# question_plot_metrics = EXP1_QUESTION_COLS + ['question_total_score']

# fig, axes = plt.subplots(2, 4, figsize=(20, 10))
# axes = axes.flatten()
# plots[f'exp1_question_bundle_llm_{analysis_suffix}'] = fig

# for i, criterion in enumerate(question_plot_metrics):
#     scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
#     create_seaborn_boxplot(
#         exp1_df,
#         'llm',
#         criterion,
#         axes[i],
#         DISPLAY_LABELS.get(criterion, criterion.title()),
#         DISPLAY_LABELS.get(criterion, criterion.title()),
#         'LLM',
#         scale_range=scale_range,
#     )

# for j in range(len(question_plot_metrics), len(axes)):
#     axes[j].set_axis_off()

# plt.suptitle('Experiment 1: Question-based Criteria', fontsize=16, fontweight='bold', y=0.98)
# plt.tight_layout()
# plt.subplots_adjust(top=0.92)
# plt.show()

In [ ]:
# fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# axes = axes.flatten()
# plots[f'exp1_answer_bundle_llm_{analysis_suffix}'] = fig

# for i, criterion in enumerate(answer_plot_metrics):
#     scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
#     create_seaborn_boxplot(
#         exp1_df,
#         'llm',
#         criterion,
#         axes[i],
#         DISPLAY_LABELS.get(criterion, criterion.title()),
#         DISPLAY_LABELS.get(criterion, criterion.title()),
#         'LLM',
#         scale_range=scale_range,
#     )

# plt.suptitle('Experiment 1: Answer-based Criteria', fontsize=16, fontweight='bold', y=0.98)
# plt.tight_layout()
# plt.subplots_adjust(top=0.92)
# plt.show()

In [ ]:
fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(2, 8)

axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[0, 4:6]),
    fig.add_subplot(gs[0, 6:8]),
    fig.add_subplot(gs[1, 1:3]),
    fig.add_subplot(gs[1, 3:5]),
    fig.add_subplot(gs[1, 5:7]),
]

plots[f'exp1_question_bundle_llm_{analysis_suffix}'] = fig

for i, criterion in enumerate(question_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp1_df,
        'llm',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'LLM',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 1: Question-based Criteria', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
plots[f'exp1_answer_bundle_llm_{analysis_suffix}'] = fig

for i, criterion in enumerate(answer_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp1_df,
        'llm',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'LLM',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 1: Answer-based Criteria', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
# fig, axes = plt.subplots(2, 4, figsize=(20, 10))
# axes = axes.flatten()
# plots[f'exp1_question_bundle_question_type_{analysis_suffix}'] = fig

# for i, criterion in enumerate(question_plot_metrics):
#     scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
#     create_seaborn_boxplot(
#         exp1_df,
#         'question_type',
#         criterion,
#         axes[i],
#         DISPLAY_LABELS.get(criterion, criterion.title()),
#         DISPLAY_LABELS.get(criterion, criterion.title()),
#         'Question Type',
#         scale_range=scale_range,
#     )

# for j in range(len(question_plot_metrics), len(axes)):
#     axes[j].set_axis_off()

# plt.suptitle('Experiment 1: Question-based Criteria', fontsize=16, fontweight='bold', y=0.98)
# plt.tight_layout()
# plt.subplots_adjust(top=0.92)
# plt.show()

In [ ]:
exp1_df_plot = exp1_df.copy()
exp1_df_plot['bloom_idx'] = pd.to_numeric(exp1_df_plot['bloom_idx'], errors='coerce')
exp1_df_plot = exp1_df_plot[exp1_df_plot['bloom_idx'].between(1, 6, inclusive='both')].copy()

LABEL_MAPPING.update({str(i): f'{i}' for i in range(1, 7)})

question_plot_metrics = EXP1_QUESTION_COLS + ['question_total_score']

fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(2, 8)

axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[0, 4:6]),
    fig.add_subplot(gs[0, 6:8]),
    fig.add_subplot(gs[1, 1:3]),
    fig.add_subplot(gs[1, 3:5]),
    fig.add_subplot(gs[1, 5:7]),
]

plots[f'exp1_question_bundle_bloom_{analysis_suffix}'] = fig

for i, criterion in enumerate(question_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp1_df_plot,
        'bloom_idx',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'Bloom Level',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 1: Question-based Criteria by Bloom Level', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
exp1_df_plot = exp1_df.copy()
exp1_df_plot['bloom_idx'] = pd.to_numeric(exp1_df_plot['bloom_idx'], errors='coerce')
exp1_df_plot = exp1_df_plot[exp1_df_plot['bloom_idx'].between(1, 6, inclusive='both')].copy()

answer_plot_metrics = EXP1_ANSWER_COLS + ['answer_total_score']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
plots[f'exp1_answer_bundle_bloom_{analysis_suffix}'] = fig

for i, criterion in enumerate(answer_plot_metrics):
    scale_range = SCORE_RANGES.get(criterion, LIKERT_RANGE)
    create_seaborn_boxplot(
        exp1_df_plot,
        'bloom_idx',
        criterion,
        axes[i],
        DISPLAY_LABELS.get(criterion, criterion.title()),
        DISPLAY_LABELS.get(criterion, criterion.title()),
        'Bloom Level',
        scale_range=scale_range,
    )

plt.suptitle('Experiment 1: Answer-based Criteria by Bloom Level', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
plots[f'exp1_question_total_heatmap_{analysis_suffix}'] = fig

create_heatmap(
    exp1_df,
    'llm',
    'question_type',
    'question_total_score',
    ax,
    'Question Total: LLM vs Question Type\nValues: Mean (Std) | Scale: 6-30 points',
    'Question Total Score',
    scale_range=SCORE_RANGES['question_total_score'],
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
plots[f'exp1_answer_total_heatmap_{analysis_suffix}'] = fig

create_heatmap(
    exp1_df,
    'llm',
    'question_type',
    'answer_total_score',
    ax,
    'Answer Total: LLM vs Question Type\nValues: Mean (Std) | Scale: 3-15 points',
    'Answer Total Score',
    scale_range=SCORE_RANGES['answer_total_score'],
)

plt.tight_layout()
plt.show()

answer_llm_source_stats = exp1_df.groupby(['llm', 'input_source'])['answer_total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_answer_llm_source_stats_{analysis_suffix}'] = answer_llm_source_stats

answer_llm_question_type_stats = exp1_df.groupby(['llm', 'question_type'])['answer_total_score'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_answer_llm_question_type_stats_{analysis_suffix}'] = answer_llm_question_type_stats

correctness_llm_question_type_stats = exp1_df.groupby(['llm', 'question_type'])['a_correctness'].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables[f'exp1_correctness_llm_question_type_stats_{analysis_suffix}'] = correctness_llm_question_type_stats

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
plots[f'exp1_total_score_heatmap_{analysis_suffix}'] = fig

create_heatmap(
    exp1_df,
    'llm',
    'question_type',
    'total_score',
    ax,
    'Total Score: LLM vs Question Type\nValues: Mean (Std) | Scale: 9-45 points',
    'Overall Total Score',
    scale_range=SCORE_RANGES['total_score'],
)

plt.tight_layout()
plt.show()

# Inter-Rater Agreement Analysis

Agreement is computed across the unified set of 3 experts. All agreement calculations use the 1-5 Likert scale from the evaluator CSVs.

In [ ]:
print("INTER-RATER AGREEMENT ANALYSIS (Fleiss' Kappa, Kendall's W, ICC(3,1))")
print("=" * 60)
print(f"Analyzing agreement across {len(EXPERT_IDS)} experts: {EXPERT_IDS}")

expert_dfs_dict = {
    expert_key: experts_data[expert_key]
    for expert_key in [f'expert_{expert_num}' for expert_num in EXPERT_IDS]
    if expert_key in experts_data
}

agreement_exp1 = calculate_agreement_reliability(
    expert_dfs_dict,
    EXP1_NUMERIC_COLS,
    LIKERT_MIN,
    LIKERT_MAX,
 )
icc_exp1 = calculate_icc_3_1(
    expert_dfs_dict,
    EXP1_NUMERIC_COLS,
    LIKERT_MIN,
    LIKERT_MAX,
)

if not icc_exp1.empty:
    agreement_exp1 = agreement_exp1.merge(
        icc_exp1[['Criterion', 'ICC3_1', 'ICC3_1_CI95', 'ICC3_1_pval']],
        on='Criterion',
        how='left',
    )

if not agreement_exp1.empty:
    agreement_exp1 = agreement_exp1.sort_values(
        'Criterion',
        key=lambda s: s.str.startswith('a_').astype(int),
        kind='stable'
    ).reset_index(drop=True)
    core_cols = ['Criterion', 'Fleiss_K', 'Fleiss_K_pval', 'Agreement', 'Kendalls_W', 'Kendalls_W_pval', 'ICC3_1', 'ICC3_1_CI95', 'ICC3_1_pval']
    ordered_cols = [c for c in core_cols if c in agreement_exp1.columns]
    remaining_cols = [c for c in agreement_exp1.columns if c not in ordered_cols]
    agreement_exp1 = agreement_exp1[ordered_cols + remaining_cols]

    tables[f'agreement_exp1_{analysis_suffix}'] = agreement_exp1
    display(agreement_exp1.round(3))

    valid_kappas = agreement_exp1['Fleiss_K'].dropna()
    valid_ws = agreement_exp1['Kendalls_W'].dropna()
    valid_icc = agreement_exp1['ICC3_1'].dropna() if 'ICC3_1' in agreement_exp1.columns else pd.Series(dtype=float)
    if len(valid_kappas) > 0:
        avg_kappa = valid_kappas.mean()
        avg_w = valid_ws.mean()
        level = (
            "Poor" if avg_kappa < 0 else
            "Slight" if avg_kappa < 0.2 else
            "Fair" if avg_kappa < 0.4 else
            "Moderate" if avg_kappa < 0.6 else
            "Substantial" if avg_kappa < 0.8 else
            "Almost Perfect"
        )
        print(f"\nAverage Fleiss' Kappa: {avg_kappa:.3f} ({level})")
        print(f"Average Kendall's W: {avg_w:.3f}")
    if len(valid_icc) > 0:
        print(f"Average ICC(3,1): {valid_icc.mean():.3f}")

print("\n" + "=" * 60)
print("EXPERT COMPARISON - INDIVIDUAL STATISTICS")
print("=" * 60)

expert_means = {}
available_expert_keys = [k for k in [f'expert_{n}' for n in EXPERT_IDS] if k in experts_data]

for expert_key in available_expert_keys:
    cleaned_expert_df = clean_numeric_data(experts_data[expert_key], EXP1_NUMERIC_COLS)
    cleaned_expert_df = calculate_exp1_scores(cleaned_expert_df)
    expert_means[expert_key] = cleaned_expert_df[EXP1_NUMERIC_COLS + EXP1_SCORE_COLS].mean()

comparison_df = pd.DataFrame(expert_means).T
comparison_df = comparison_df.round(3)
tables[f'expert_comparison_exp1_{analysis_suffix}'] = comparison_df

print("\nMean Ratings by Expert (Experiment 1):")
display(comparison_df)

In [ ]:
if 'agreement_exp1' in locals() and not agreement_exp1.empty:
    summary_cols = ['Fleiss_K', 'Kendalls_W', 'Mean', 'Std']
    if 'ICC3_1' in agreement_exp1.columns:
        summary_cols.append('ICC3_1')
    agreement_summary = agreement_exp1.groupby('Domain')[summary_cols].mean().round(3)
    tables[f'agreement_summary_exp1_{analysis_suffix}'] = agreement_summary
    print("Average agreement by domain:")
    display(agreement_summary)
else:
    print("No agreement summary available.")

## Data Export

Save all tables and plots for further analysis and reporting.

In [ ]:
# Save all tables and plots
for table_name, table_df in tables.items():
    if isinstance(table_df, pd.DataFrame):
        safe_name = table_name.replace(' ', '_').replace('(', '').replace(')', '').replace("'", '').lower()
        csv_path = os.path.join(output_tables_path, f'{safe_name}.csv')
        table_df.to_csv(csv_path)
        print(f'Saved table: {csv_path}')

for plot_name, fig in plots.items():
    safe_name = plot_name.replace(' ', '_').replace('(', '').replace(')', '').replace("'", '').lower()
    png_path = os.path.join(output_plots_path, f'{safe_name}.png')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    print(f'Saved plot: {png_path}')

print(f'\nAnalysis complete. Results saved to: {output_base_path}')